[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/natrask/AESCAPE/blob/main/notebooks/01_react.ipynb)

# Part 1 — ReAct

> **Control flow: the model.** At each step it looks at everything so far and picks one action.

ReAct (Reason + Act, [Yao et al. 2022](https://arxiv.org/abs/2210.03629)) is the baseline every
other pattern is a reaction to. There is no framework — it is a `for` loop and a dispatch table.

In [ ]:
# Setup -- same as 00_api_access.ipynb
import os, sys, time, json
if 'google.colab' in sys.modules:
    %pip install -U -q google-genai

from google import genai
from google.genai import types as gtypes

try:
    from google.colab import userdata
    API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    API_KEY = os.environ.get('GEMINI_API_KEY', '')

assert API_KEY, "Set GEMINI_API_KEY in Colab secrets or your environment."

client = genai.Client(api_key=API_KEY)
MODEL = "gemini-3.1-flash-lite"


def generate_with_retry(*, contents, config=None, max_attempts=6):
    delay = 4.0
    for attempt in range(max_attempts):
        try:
            return client.models.generate_content(model=MODEL, contents=contents, config=config)
        except Exception as e:
            msg = str(e)
            if "429" not in msg and "RESOURCE_EXHAUSTED" not in msg and "quota" not in msg.lower():
                raise
            if attempt == max_attempts - 1:
                raise
            print(f"[rate limit] attempt {attempt+1}: sleeping {delay:.1f}s and retrying...")
            time.sleep(delay)
            delay = min(delay * 2, 60.0)

print(f"Gemini client ready (model={MODEL}).")

### 1.1 A tool is a function plus a schema

Two pieces, both written by you: an ordinary Python function, and a JSON-schema description of
its signature that you hand to the model.

The schema is not bookkeeping — it is prompt engineering. `name` and `description` decide whether
the model picks *this* tool over the others, and every value you forbid with `enum` is a failure
mode you never have to debug.

In [ ]:
def calculator(op: str, a: float, b: float) -> dict:
    """Perform one arithmetic operation."""
    ops = {"add": a + b, "sub": a - b, "mul": a * b,
           "div": a / b if b != 0 else float('nan')}
    if op not in ops:
        return {"error": f"unknown op {op!r}; valid: add, sub, mul, div"}
    return {"result": ops[op]}


CALC_PARAMS = {
    "type": "object",
    "properties": {
        "op": {"type": "string", "enum": ["add", "sub", "mul", "div"],
               "description": "operation to perform"},
        "a":  {"type": "number"},
        "b":  {"type": "number"},
    },
    "required": ["op", "a", "b"],
}

def calc_schema():
    return gtypes.FunctionDeclaration(
        name="calculator",
        description="Perform one arithmetic operation.",
        parameters=CALC_PARAMS)

print(json.dumps(CALC_PARAMS, indent=2))

### 1.2 The loop

Three moving parts:

1. Call the model with the running conversation and the tool schemas.
2. If the response contains a `function_call`, run the tool and append its output as an observation.
3. If the response is plain text, that is the final answer.

Tool dispatch is wrapped in `try/except` so a malformed call comes back to the model as an
observation instead of killing the run. **You will see this trigger.** An agent recovering from
its own bad tool call is the system working.

In [ ]:
def run_agent(system_prompt, user_prompt, tool_schemas, tool_fns,
              max_steps=12, temperature=0.2, verbose=True):
    """Minimal ReAct loop. Returns (final_text, transcript)."""
    contents = [gtypes.Content(role="user",
                               parts=[gtypes.Part.from_text(text=user_prompt)])]
    cfg = gtypes.GenerateContentConfig(
        system_instruction=system_prompt,
        tools=[gtypes.Tool(function_declarations=tool_schemas)],
        temperature=temperature,
        automatic_function_calling=gtypes.AutomaticFunctionCallingConfig(disable=True))

    transcript = []
    for step in range(max_steps):
        resp = generate_with_retry(contents=contents, config=cfg)
        parts = resp.candidates[0].content.parts or []
        contents.append(resp.candidates[0].content)

        calls = [p.function_call for p in parts if getattr(p, "function_call", None)]
        if not calls:
            text = "".join(getattr(p, "text", "") or "" for p in parts)
            transcript.append(("final", text))
            if verbose: print(f"[{step}] FINAL: {text[:160]}")
            return text, transcript

        obs = []
        for fc in calls:
            name, args = fc.name, dict(fc.args or {})
            if verbose: print(f"[{step}] CALL {name}({args})")
            try:
                result = tool_fns[name](**args)
            except Exception as e:
                result = {"error": f"{type(e).__name__}: {e}"}   # errors are observations
            transcript.append((name, args, result))
            if verbose: print(f"[{step}]   -> {json.dumps(result)[:160]}")
            obs.append(gtypes.Part.from_function_response(name=name, response=result))
        contents.append(gtypes.Content(role="user", parts=obs))

    transcript.append(("final", "(max_steps reached)"))
    return "(max_steps reached)", transcript

### 1.3 The toy task

Every pattern in this notebook gets the *same* trivial job, so you are comparing syntax rather
than problems: **compute `13 * 47 + 8` using the calculator tool.** The answer is 619.

The toy is deliberately too easy to justify any of these architectures. That is the point — with
the problem out of the way you can see the raw shape of each framework.

In [ ]:
TOY_SYSTEM = ("You are a careful arithmetic assistant. Use the calculator tool for every "
              "computation; never do arithmetic in your head.")
TOY_USER = "Compute 13 * 47 + 8. Return just the final number."

answer, transcript = run_agent(TOY_SYSTEM, TOY_USER,
                               tool_schemas=[calc_schema()],
                               tool_fns={"calculator": calculator},
                               max_steps=8)
print("\n=== Final answer:", answer)

### 1.4 When to reach for ReAct

**Use it when** the procedure cannot be specified in advance, mistakes are cheap and recoverable,
and the tool surface is small.

**Its exemplar** is adaptive mesh refinement (notebook `02_agent_hackathon.ipynb`): the number of
refinement cycles depends on error you have not measured yet, a bad refinement just produces a
mesh you discard, and the model must react to observations it could not have predicted. Greedy,
one-step-at-a-time decision making is not a compromise there — it is the correct algorithm.

**Where it starts to hurt** is everything in Parts 2–4.